# Data Processing

This notebook splices the raw Tynitag NTC logger CSV files into continuous timeseries for each borehole.

The Tynitag loggers are read out periodically; each read produces a new CSV file that begins at the download time. This notebook loads all individual fragments per borehole, applies 0-degree calibration offsets, and concatenates them into a single spliced timeseries file that is then used by the analysis notebooks.

**Output:** one `*_spliced.csv` file per borehole under `results/`.


## 1. Import required Libraries and Modules

In [1]:
import sys
import os
import pandas as pd

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.thermistor_processing import *


## 2. Set data and output paths

In [2]:
# main temperature data path
main_dir = '/Users/janoschbeer/Library/Mobile Documents/com~apple~CloudDocs/PhD/data/fieldwork_data/THERMAP_2024_2025/icetemperature_data/NTC_tynitag/'
temp_path = main_dir + 'temperature_data/'
first_timeseries = '2024/'
second_timeseries = '2024_2025/'

# set output path
output_path = main_dir + 'temperature_data/full_timeseries/'

# Hohsaasgletscher
hohsaas_1st_timeseries = [temp_path + first_timeseries + 'BH5_20240808_20240929.csv', temp_path + first_timeseries + 'BH6_20240808_20240929.csv']
hohsaas_2nd_timeseries = [temp_path + second_timeseries + 'BH5_20240929_20250927.csv', temp_path + second_timeseries + 'BH6_20240929_20250927.csv']

# Chessjengletscher
chessjen_1st_timeseries = [temp_path + first_timeseries + 'BH7_20240809_20240929.csv', temp_path + first_timeseries + 'BH8_20240809_20240929.csv']
chessjen_2nd_timeseries = [temp_path + second_timeseries + 'BH7_20240929_20250808.csv', temp_path + second_timeseries + 'BH8_20240929_20250808.csv']

# Alphubel South
alphubel_1st_timeseries = temp_path + first_timeseries + 'BH10_20240821_20241021.csv'
alphubel_2nd_timeseries = temp_path + second_timeseries + 'BH10_20241021_20250805.csv'
alphubel_3rd_timeseries = temp_path + second_timeseries + 'BH10_20250805_20250916.csv' # 3 readings, borehole 9 read only ones

# Sex Rouge
sex_rouge_1st_timeseries = [temp_path + first_timeseries + 'BH1_20240806_20241019.csv', temp_path + first_timeseries + 'BH2_20240806_20241019.csv']
sex_rouge_2nd_timeseries = [temp_path + second_timeseries + 'BH1_20240930_20250724.csv', temp_path + second_timeseries + 'BH2_20240930_20250724.csv']

# Glacier de Tortin
tortin_1st_timeseries = [temp_path + first_timeseries + 'BH3_20240807_20240930.csv', temp_path + first_timeseries + 'BH4_20240807_20240930.csv']
tortin_2nd_timeseries = [temp_path + second_timeseries + 'BH3_20240939_20250723.csv', temp_path + second_timeseries + 'BH4_20240930_20250723.csv']

# Corvatsch
corvatsch_1st_timeseries = temp_path + first_timeseries + 'BH12_20240828_20241020.csv'
corvatsch_2nd_timeseries = temp_path + second_timeseries + 'BH12_20241123_20250905.csv' # borehole 11 read only ones

# read offsets from ice bath references
offsets_path_TT = main_dir + "calibration_data/all_logger_offsets.csv"
corrected_offsets_TT = pd.read_csv(offsets_path_TT)

## 3. Load borehole data for each borehole and each timeseries & apply 0-deg offsets

In [3]:
## Create ThermistorData objects for each borehole and timeseries

# Hohsaasgletscher
hs1tt_1st = ThermistorData(hohsaas_1st_timeseries[0], delimiter=',') # e.g. hs1tt_1st means Hohsaas borehole 1, Tynitag, first timeseries (08.2024 - 09.2024)
hs2tt_1st = ThermistorData(hohsaas_1st_timeseries[1], delimiter=',')
hs1tt_2nd = ThermistorData(hohsaas_2nd_timeseries[0], delimiter=',')
hs2tt_2nd = ThermistorData(hohsaas_2nd_timeseries[1], delimiter=',')

# Chessjengletscher
cj1tt_1st = ThermistorData(chessjen_1st_timeseries[0], delimiter=',')
cj2tt_1st = ThermistorData(chessjen_1st_timeseries[1], delimiter=',')
cj1tt_2nd = ThermistorData(chessjen_2nd_timeseries[0], delimiter=',')
cj2tt_2nd = ThermistorData(chessjen_2nd_timeseries[1], delimiter=',')

# Alphubel South
ah1tt_1st = ThermistorData(alphubel_1st_timeseries, delimiter=',')
ah1tt_2nd = ThermistorData(alphubel_2nd_timeseries, delimiter=',')
ah1tt_3rd = ThermistorData(alphubel_3rd_timeseries, delimiter=',')

# Sex Rouge
sr1tt_1st = ThermistorData(sex_rouge_1st_timeseries[0], delimiter=',')
sr2tt_1st = ThermistorData(sex_rouge_1st_timeseries[1], delimiter=',')
sr1tt_2nd = ThermistorData(sex_rouge_2nd_timeseries[0], delimiter=',')
sr2tt_2nd = ThermistorData(sex_rouge_2nd_timeseries[1], delimiter=',')

# Glacier de Tortin
gt1tt_1st = ThermistorData(tortin_1st_timeseries[0], delimiter=',')
gt2tt_1st = ThermistorData(tortin_1st_timeseries[1], delimiter=',')
gt1tt_2nd = ThermistorData(tortin_2nd_timeseries[0], delimiter=',')
gt2tt_2nd = ThermistorData(tortin_2nd_timeseries[1], delimiter=',')

# Corvatsch
ct2tt_1st = ThermistorData(corvatsch_1st_timeseries, delimiter=',')
ct2tt_2nd = ThermistorData(corvatsch_2nd_timeseries, delimiter=',')

## Get NTC data with offsets applied

# Hohsaasgletscher
hs1tt_1st_data = hs1tt_1st.get_ntc_data_with_offsets('5', offsets_df=corrected_offsets_TT)
hs2tt_1st_data = hs2tt_1st.get_ntc_data_with_offsets('6', offsets_df=corrected_offsets_TT)
hs1tt_2nd_data = hs1tt_2nd.get_ntc_data_with_offsets('5', offsets_df=corrected_offsets_TT)
hs2tt_2nd_data = hs2tt_2nd.get_ntc_data_with_offsets('6', offsets_df=corrected_offsets_TT)

# Chessjengletscher
cj1tt_1st_data = cj1tt_1st.get_ntc_data_with_offsets('7', offsets_df=corrected_offsets_TT)
cj2tt_1st_data = cj2tt_1st.get_ntc_data_with_offsets('8', offsets_df=corrected_offsets_TT)
cj1tt_2nd_data = cj1tt_2nd.get_ntc_data_with_offsets('7', offsets_df=corrected_offsets_TT)
cj2tt_2nd_data = cj2tt_2nd.get_ntc_data_with_offsets('8', offsets_df=corrected_offsets_TT)

# Alphubel South
ah1tt_1st_data = ah1tt_1st.get_ntc_data_with_offsets('10', offsets_df=corrected_offsets_TT)
ah1tt_2nd_data = ah1tt_2nd.get_ntc_data_with_offsets('10', offsets_df=corrected_offsets_TT)
ah1tt_3rd_data = ah1tt_3rd.get_ntc_data_with_offsets('10', offsets_df=corrected_offsets_TT)

# Sex Rouge
sr1tt_1st_data = sr1tt_1st.get_ntc_data_with_offsets('1', offsets_df=corrected_offsets_TT)
sr2tt_1st_data = sr2tt_1st.get_ntc_data_with_offsets('2', offsets_df=corrected_offsets_TT)
sr1tt_2nd_data = sr1tt_2nd.get_ntc_data_with_offsets('1', offsets_df=corrected_offsets_TT)
sr2tt_2nd_data = sr2tt_2nd.get_ntc_data_with_offsets('2', offsets_df=corrected_offsets_TT)

# Glacier de Tortin
gt1tt_1st_data = gt1tt_1st.get_ntc_data_with_offsets('3', offsets_df=corrected_offsets_TT)
gt2tt_1st_data = gt2tt_1st.get_ntc_data_with_offsets('4', offsets_df=corrected_offsets_TT)
gt1tt_2nd_data = gt1tt_2nd.get_ntc_data_with_offsets('3', offsets_df=corrected_offsets_TT)
gt2tt_2nd_data = gt2tt_2nd.get_ntc_data_with_offsets('4', offsets_df=corrected_offsets_TT)

# Corvatsch
ct2tt_1st_data = ct2tt_1st.get_ntc_data_with_offsets('12', offsets_df=corrected_offsets_TT)
ct2tt_2nd_data = ct2tt_2nd.get_ntc_data_with_offsets('12', offsets_df=corrected_offsets_TT)

## 4. Splice dataframes together & export files containing full tynitag data timeseries

In [4]:
# Splice Hohsaas boreholes HS1 (BH5) and HS2 (BH6) and export
hs1tt_spliced = splice_timeseries(
    [hs1tt_1st_data, hs1tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "HS1TT_20240808_20250927_spliced.csv",
    file_stem="HS1TT_20240808_20250927_spliced"
)

hs2tt_spliced = splice_timeseries(
    [hs2tt_1st_data, hs2tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "HS2TT_20240808_20250927_spliced.csv",
    file_stem="HS2TT_20240808_20250927_spliced"
)

# Splice Chessjen boreholes CJ1 (BH7) and CJ2 (BH8) and export
cj1tt_spliced = splice_timeseries(
    [cj1tt_1st_data, cj1tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "CJ1TT_20240809_20250808_spliced.csv",
    file_stem="CJ1TT_20240809_20250808_spliced"
)

cj2tt_spliced = splice_timeseries(
    [cj2tt_1st_data, cj2tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "CJ2TT_20240809_20250808_spliced.csv",
    file_stem="CJ2TT_20240809_20250808_spliced"
)

# Splice Glacier de Tortin boreholes GT1 (BH3) and GT2 (BH4) and export
gt1tt_spliced = splice_timeseries(
    [gt1tt_1st_data, gt1tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "GT1TT_20240807_20250723_spliced.csv",
    file_stem="GT1TT_20240807_20250723_spliced"
)

gt2tt_spliced = splice_timeseries(
    [gt2tt_1st_data, gt2tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "GT2TT_20240807_20250723_spliced.csv",
    file_stem="GT2TT_20240807_20250723_spliced"
)

# Splice Sex Rouge boreholes SR1 (BH1) and SR2 (BH2) and export
sr1tt_spliced = splice_timeseries(
    [sr1tt_1st_data, sr1tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "SR1TT_20240806_20250724_spliced.csv",
    file_stem="SR1TT_20240806_20250724_spliced"
) 

sr2tt_spliced = splice_timeseries(
    [sr2tt_1st_data, sr2tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "SR2TT_20240806_20250724_spliced.csv",
    file_stem="SR2TT_20240806_20250724_spliced"
)

# Splice Corvatsch borehole CT2 (BH12) and export
ct2tt_spliced = splice_timeseries(
    [ct2tt_1st_data, ct2tt_2nd_data],
    time_col="TIME",
    out_path=output_path + "CT2TT_20240828_20250905_spliced.csv",
    file_stem="CT2TT_20240828_20250905_spliced"
)

# Splice Alphubel South borehole AH1 (BH10) and export
ah1tt_spliced = splice_timeseries(
    [ah1tt_1st_data, ah1tt_2nd_data, ah1tt_3rd_data],
    time_col="TIME",
    out_path=output_path + "AH1TT_20240821_20250916_spliced.csv",
    file_stem="AH1TT_20240821_20250916_spliced"
)

